# NB6 - Feature Extraction: Anomaly + Context -> 42-dim Vectors (Person B)
**Run this AFTER NB5 is complete and `mentalmanip-vlits` is shared as a Kaggle dataset.**

## Setup:
1. `+ Add Data` -> search `mentalmanip-vlits` -> add it
2. GPU not required
3. Run all cells

Output: `mentalmanip_features.csv` -> share as Kaggle dataset `mentalmanip-features`
Rows from MUSTARD are carried through and labeled as class 1 (Sarcasm).
Person D adds this as input to NB7.


## 0. Install

In [ ]:
%%capture
!pip install numpy pandas scikit-learn scipy -q
print('done')

## 1. Config

In [ ]:
import os
import re
import numpy as np
import pandas as pd

VLITS_CSV = '/kaggle/input/mentalmanip-vlits/mentalmanip_vlits.csv'
OUT_DIR   = '/kaggle/working'
EMOTIONS  = ['joy','trust','fear','surprise','sadness','disgust','anger','anticipation']
# Plutchik opposite pairs (for tension scores)
OPPOSITE_PAIRS = [(0,4),(1,5),(2,6),(3,7)]  # joy/sadness, trust/disgust, fear/anger, surprise/anticipation
N_EMO = 8
EMA_ALPHA = 0.3    # context tracker decay
print('Config ready')

## 2. Load V_lit Data

In [ ]:
df = pd.read_csv(VLITS_CSV)
if 'source_dataset' not in df.columns:
    df['source_dataset'] = 'mentalmanip'
if 'annotator_votes' not in df.columns:
    # backward compat: assume all rows are high-confidence
    df['annotator_votes'] = 3
print(f'Loaded: {len(df):,} rows | Columns: {list(df.columns)}')
print(f'Dialogues: {df["dialogue_id"].nunique():,}')
print(df['source_dataset'].value_counts(dropna=False))
print(f'annotator_votes dist: {dict(df["annotator_votes"].value_counts().sort_index())}')
print(df.head(3))


## 3. Anomaly Detection Functions

In [ ]:
def spike_score(v):
    """How dominant is the top emotion? High = clear single emotion."""
    return float(np.max(v) - np.mean(v))


def tension_score(v):
    """
    Sum of product co-activations on all 4 Plutchik opposite pairs.
    Product formula (v[i]*v[j]) fires whenever both emotions are active,
    unlike the thresholded sum which needs both > 0.5 simultaneously.
    """
    total = 0.0
    for i, j in OPPOSITE_PAIRS:
        total += float(v[i]) * float(v[j])
    return total


def suppression_score(v):
    """All emotions low → emotionally flat/suppressed."""
    return float(1.0 - np.max(v)) if np.max(v) < 0.3 else 0.0


def incoherence_score(v):
    """Entropy of emotion distribution (normalised 0-1)."""
    v = np.array(v, dtype=np.float32) + 1e-8
    normed = v / v.sum()
    return float(-np.sum(normed * np.log(normed)) / np.log(N_EMO))


print('Anomaly functions ready')


In [ ]:
## 3b. Lexical Feature Functions
# 4 pragmatic signals that are invisible to the emotion vector alone.
# These are derived directly from the raw utterance text.

_NEGATION = re.compile(
    r"\b(not|never|no\b|don't|doesn't|didn't|can't|won't|isn't|aren't|wasn't|weren't"
    r"|nothing|nobody|nowhere|neither|nor)\b",
    re.IGNORECASE
)
_PRESSURE = re.compile(
    r"\b(everyone|everybody|always|supposed|should|must|expected|assumed|awkward"
    r"|obviously|clearly|of\s+course|you\s+always|you\s+never|why\s+can't\s+you"
    r"|what's\s+wrong\s+with\s+you|disappointed|I\s+assumed|I\s+expected)\b",
    re.IGNORECASE
)
_HEDGING = re.compile(
    r"\b(maybe|perhaps|possibly|probably|might|could\s+be|seems|apparently"
    r"|I\s+think|I\s+feel|I\s+guess|sort\s+of|kind\s+of|I\s+suppose|I\s+believe)\b",
    re.IGNORECASE
)


def lexical_features(text: str) -> np.ndarray:
    """4 features capturing pragmatic signals not present in emotion vectors.

    - lex_negation: density of negation words — sarcasm/gaslighting signal
    - lex_pressure: density of universalising/pressure words — manipulation signal
    - lex_hedging:  density of hedging language — genuine speech signal
    - lex_punct:    1 if text contains ! or ?, else 0 — emotional intensity signal
    """
    text = str(text)
    words = text.split()
    n = max(len(words), 1)
    negation = min(len(_NEGATION.findall(text)) / n, 1.0)
    pressure = min(len(_PRESSURE.findall(text)) / n, 1.0)
    hedging  = min(len(_HEDGING.findall(text))  / n, 1.0)
    punct    = float(bool(re.search(r'[!?]', text)))
    return np.array([negation, pressure, hedging, punct], dtype=np.float32)


print('Lexical feature functions ready  (4 features: negation, pressure, hedging, punct)')

## 4. Context Tracker (Rolling EMA per Speaker)

In [ ]:
class ConversationTracker:
    """
    Maintains rolling EMA context vector per speaker.
    Also tracks the last 3 conversation-level context vectors for delta computation.
    """
    def __init__(self, alpha=EMA_ALPHA):
        self.alpha      = alpha
        self.v_ctxt     = np.zeros(N_EMO)
        self.v_speakers = {}
        self.history    = []

    def update(self, v_lit, speaker):
        self.history.append(self.v_ctxt.copy())
        if len(self.history) > 3:
            self.history.pop(0)
        self.v_ctxt = self.alpha * v_lit + (1 - self.alpha) * self.v_ctxt
        if speaker not in self.v_speakers:
            self.v_speakers[speaker] = v_lit.copy()
        else:
            self.v_speakers[speaker] = (self.alpha * v_lit
                                        + (1 - self.alpha) * self.v_speakers[speaker])

    def get_deltas(self, v_lit):
        h = self.history
        dv1 = v_lit - h[-1] if len(h) >= 1 else np.zeros(N_EMO)
        dv2 = v_lit - h[-2] if len(h) >= 2 else np.zeros(N_EMO)
        dv3 = v_lit - h[-3] if len(h) >= 3 else np.zeros(N_EMO)
        return dv1, dv2, dv3

    def drift_score(self, v_lit):
        return float(np.linalg.norm(v_lit - self.v_ctxt))

    def actor_specificity(self, v_lit, speaker):
        spk_ctxt = self.v_speakers.get(speaker, self.v_ctxt)
        return float(np.linalg.norm(spk_ctxt - self.v_ctxt))


print('ConversationTracker ready')

In [ ]:
FEATURE_NAMES = (
    EMOTIONS +
    [f'dv1_{e}' for e in EMOTIONS] +
    [f'dv2_{e}' for e in EMOTIONS] +
    [f'dv3_{e}' for e in EMOTIONS] +
    ['tension_joy_sadness','tension_trust_disgust','tension_fear_anger','tension_surprise_anticip'] +
    ['drift','actor_specificity','spike','tension_total','suppression','incoherence'] +
    ['lex_negation','lex_pressure','lex_hedging','lex_punct']
)
assert len(FEATURE_NAMES) == 46, f'Expected 46 features, got {len(FEATURE_NAMES)}'
print(f'Feature vector: {len(FEATURE_NAMES)} dims')


def extract_features(v_lit, tracker, speaker, text: str = ''):
    dv1, dv2, dv3 = tracker.get_deltas(v_lit)
    pair_tensions = [float(v_lit[i]) * float(v_lit[j]) for i, j in OPPOSITE_PAIRS]
    lex = lexical_features(text)
    return np.concatenate([
        v_lit,
        dv1, dv2, dv3,
        pair_tensions,
        [tracker.drift_score(v_lit),
         tracker.actor_specificity(v_lit, speaker),
         spike_score(v_lit),
         tension_score(v_lit),
         suppression_score(v_lit),
         incoherence_score(v_lit)],
        lex,
    ]).astype(np.float32)

In [ ]:
rows = []

for dialogue_id, group in df.groupby('dialogue_id', sort=False):
    group   = group.sort_values('turn_index').reset_index(drop=True)
    tracker = ConversationTracker()

    for _, turn in group.iterrows():
        v_lit   = turn[EMOTIONS].values.astype(np.float32)
        speaker = str(turn['speaker'])
        text    = str(turn.get('text', ''))
        feats   = extract_features(v_lit, tracker, speaker, text=text)
        tracker.update(v_lit, speaker)

        row = {
            'dialogue_id':     dialogue_id,
            'turn_index':      turn['turn_index'],
            'speaker':         speaker,
            'is_manipulative': int(turn['is_manipulative']),
            'technique':       turn.get('technique', ''),
            'source_dataset':  turn.get('source_dataset', 'mentalmanip'),
            'annotator_votes': int(turn.get('annotator_votes', 3)),
        }
        for name, val in zip(FEATURE_NAMES, feats):
            row[name] = float(val)
        rows.append(row)

feat_df = pd.DataFrame(rows)
print(f'Feature matrix: {feat_df.shape}')
print(f'NaN count: {feat_df[FEATURE_NAMES].isna().sum().sum()}')
print(feat_df[FEATURE_NAMES].describe().round(3))

## 7. Derive 4-class Label
Classes: 0=Genuine, 1=Sarcasm, 2=Manipulation, 3=Ambiguous

In [ ]:
# Sources that are confirmed sarcasm/irony datasets
SARCASM_SOURCES    = {'mustard', 'tweet_eval_irony', 'sarcasm_headlines'}
SARCASM_TECHNIQUES = {'sarcasm', 'irony', 'mocking', 'rhetorical'}


def assign_label(row):
    source = str(row.get('source_dataset', 'mentalmanip')).lower()
    tech   = str(row.get('technique', '')).lower().strip()
    votes  = int(row.get('annotator_votes', 3))

    # Dedicated sarcasm/irony dataset → Sarcasm (regardless of other fields)
    if source in SARCASM_SOURCES:
        return 1

    # Annotator disagreement (only 1 of 3 flagged it) → Ambiguous
    # These are genuinely borderline cases — the experts couldn't agree
    if votes == 1:
        return 3

    # Clear non-manipulation → Genuine
    if not row['is_manipulative']:
        return 0

    # MentalManip row labelled with a sarcasm technique → Sarcasm
    if any(s in tech for s in SARCASM_TECHNIQUES):
        return 1

    # Clear manipulation with a known technique → Manipulation
    if tech and tech not in ('', 'nan', 'none'):
        return 2

    # Manipulative but technique unidentified → Ambiguous
    return 3


feat_df['label'] = feat_df.apply(assign_label, axis=1)
CLASS_NAMES = ['Genuine', 'Sarcasm', 'Manipulation', 'Ambiguous']
print('Label distribution:')
for i, name in enumerate(CLASS_NAMES):
    n = (feat_df['label'] == i).sum()
    print(f'  {i} {name:<15}: {n:,} ({100*n/len(feat_df):.1f}%)')
print()
print(feat_df.groupby(['source_dataset', 'label']).size().to_string())


## 8. Save

In [ ]:
out_path = os.path.join(OUT_DIR, 'mentalmanip_features.csv')
feat_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)')
print(f'Rows: {len(feat_df):,}  |  Features: {len(FEATURE_NAMES)}  |  Label col: label')

import shutil
zip_path = shutil.make_archive(os.path.join(OUT_DIR, 'mentalmanip_features'), 'zip',
                                OUT_DIR, 'mentalmanip_features.csv')
print(f'Zipped: {zip_path}')
print('Share mentalmanip_features.csv as Kaggle dataset: mentalmanip-features')
print('Person D adds it as input to NB7.')
